# 02 · The bitemporal model

**Question:** what did the market know about a company's numbers *on a given date*, and how did that change when a later filing restated them?

Notebook 01 showed why one number needs two time axes. This notebook loads it into Postgres and answers point-in-time questions in SQL.

Prerequisite: `docker compose up -d`, then `python -m edgar_bitemporal.load --raw data/raw`.

In [1]:
import os, json, pathlib, datetime as dt
import pandas as pd
from IPython.display import Code
from edgar_bitemporal.db import q, connect, SQL_DIR
from edgar_bitemporal.load import load_payload

pd.set_option("display.width", 170); pd.set_option("display.max_columns", 30)
RAW = pathlib.Path(os.environ.get("EDGAR_RAW_DIR", "../data/raw"))

## The schema

One immutable row per *report* of a fact. The invariants are enforced by the database, not by the loader:
a unique key on (company, tag, unit, period, filing), and triggers that block `UPDATE`, `DELETE` and `TRUNCATE`.

In [2]:
Code(filename=str(SQL_DIR / "01_schema.sql"), language="sql")

-- Bitemporal store for SEC EDGAR XBRL "company facts".
--
-- Every number a company reports has TWO time axes:
--   valid time        = the period the number describes        (period_start .. period_end)
--   transaction time  = when the market could first see it     (filed, the filing date)
-- A later filing can restate an earlier period. We never overwrite: each report of a fact is
-- its own immutable row, keyed by the filing (accn) that carried it. "What did we know on
-- date D?" is then a query, not a backup restore.
--
-- Invariants live in the database, not in the loader:
--   1. a given filing reports a given (company, tag, unit, period) at most once  (unique key)
--   2. rows are append-only: no UPDATE, DELETE or TRUNCATE (corrections arrive as new filings)
--
-- Deliberately NOT enforced as constraints: date sanity (filed >= period_end, period length).
-- Real filings violate them, and rejecting those rows would hide exactly the data we want to
-- study. They are monitored in checks.sql instead.

create table company (
    cik          bigint primary key,
    entity_name  text   not null
);

create table fact (
    fact_id        bigserial primary key,
    cik            bigint  not null references company (cik),
    taxonomy       text    not null,            -- us-gaap, dei, ...
    tag            text    not null,            -- XBRL concept, e.g. NetIncomeLoss
    unit           text    not null,            -- USD, shares, USD/shares, ...
    period_start   date,                        -- NULL for point-in-time (instant) concepts
    period_end     date    not null,            -- valid time
    value          numeric not null,
    accn           text    not null,            -- accession number of the reporting filing
    form           text    not null,            -- 10-K, 10-Q, ...
    filed          date    not null,            -- transaction time
    fiscal_year    int,                         -- NB: describes the FILING, not this period
    fiscal_period  text,                        -- FY, Q1, Q2, Q3 -- same caveat
    frame          text,                        -- SEC's calendarised period label, if assigned
    constraint fact_natural_key
        unique nulls not distinct (cik, taxonomy, tag, unit, period_start, period_end, accn)
);

create index fact_lookup on fact (cik, tag, unit, period_end);
create index fact_filed  on fact (filed);

-- ---- invariant 2: append-only -----------------------------------------------------------
create or replace function block_mutation() returns trigger as $$
begin
    raise exception 'fact is append-only: % not allowed (a correction arrives as a new filing)', tg_op;
end;
$$ language plpgsql;

create trigger fact_no_update_delete
    before update or delete on fact
    for each row execute function block_mutation();

create trigger fact_no_truncate
    before truncate on fact
    for each statement execute function block_mutation();

In [3]:
q('''
    select c.entity_name, count(*) as facts, count(distinct f.tag) as tags,
           min(f.filed) as first_filed, max(f.filed) as last_filed
    from fact f join company c using (cik)
    group by c.entity_name order by facts desc
''')

,entity_name,facts,tags,first_filed,last_filed
0,THE BOEING COMPANY,2914,14,2009-07-22,2026-07-28


## One restatement, end to end

Take the largest relative revision on a headline number. First, its raw history: every filing that reported that period, in the order the market saw them:

In [4]:
rev = q('''
    select * from fact_revisions
    where tag in ('Assets', 'StockholdersEquity', 'NetIncomeLoss')
    order by abs(pct_change) desc nulls last limit 1
''').iloc[0]
rev.to_frame("value")

,value
cik,12927
taxonomy,us-gaap
tag,StockholdersEquity
unit,USD
period_start,None
period_end,2017-12-31
prev_accn,0000012927-18-000007
prev_filed,2018-02-12
prev_value,355000000
accn,0000012927-18-000018


In [5]:
key = dict(cik=int(rev.cik), tag=rev.tag, unit=rev.unit,
           period_start=rev.period_start, period_end=rev.period_end)
q('''
    select filed, form, accn, value
    from fact
    where cik = %(cik)s and tag = %(tag)s and unit = %(unit)s
      and period_end = %(period_end)s
      and period_start is not distinct from %(period_start)s
    order by filed, accn
''', key)

,filed,form,accn,value
0,2018-02-12,10-K,0000012927-18-000007,355000000
1,2018-04-25,10-Q,0000012927-18-000018,1656000000
2,2018-07-25,10-Q,0000012927-18-000048,1656000000
3,2018-10-24,10-Q,0000012927-18-000065,1656000000
4,2019-02-08,10-K,0000012927-19-000010,1656000000


## Point-in-time: `facts_as_of(date)`

For each period, the value from the latest filing made on or before that date. Same period, four different "today"s:

In [6]:
one_day = dt.timedelta(days=1)
dates = [rev.prev_filed - one_day, rev.prev_filed, rev.filed, dt.date.today()]
frames = []
for d in dates:
    f = q('''
        select %(d)s::date as known_on, tag, period_end, value, form, filed as from_filing
        from facts_as_of(%(d)s)
        where cik = %(cik)s and tag = %(tag)s and unit = %(unit)s and period_end = %(period_end)s
          and period_start is not distinct from %(period_start)s
    ''', {**key, "d": d})
    if f.empty:
        f = pd.DataFrame([{"known_on": d, "tag": key["tag"], "period_end": key["period_end"],
                           "value": None, "form": None, "from_filing": None}])
    frames.append(f)
pd.concat(frames, ignore_index=True)

,known_on,tag,period_end,value,form,from_filing
0,2018-02-11,StockholdersEquity,2017-12-31,None,None,None
1,2018-02-12,StockholdersEquity,2017-12-31,355000000,10-K,2018-02-12
2,2018-04-25,StockholdersEquity,2017-12-31,1656000000,10-Q,2018-04-25
3,2026-10-10,StockholdersEquity,2017-12-31,1656000000,10-K,2019-02-08


The first row has no value at all: on that day the period had ended but nothing had been filed, so it did not exist as far as the market knew. A query that only filters on `period_end` would return the latest value there. That is look-ahead bias, and notebook 03 measures it.

## Guarantees, demonstrated

Append-only:

In [7]:
conn = connect()
try:
    with conn.cursor() as cur:
        cur.execute("update fact set value = 0 where fact_id = (select min(fact_id) from fact)")
except Exception as e:
    print(type(e).__name__, "->", str(e).strip().splitlines()[0])
finally:
    conn.rollback(); conn.close()

RaiseException -> fact is append-only: UPDATE not allowed (a correction arrives as a new filing)


Idempotent load. Re-loading the same files adds nothing (the unique key does the de-duplication):

In [8]:
conn = connect()
for f in sorted(RAW.glob("CIK*.json")):
    payload = json.loads(f.read_text())
    with conn.cursor() as cur:
        inserted, skipped = load_payload(cur, payload)
    print(f"{payload['entityName']:<40} inserted {inserted:>6,}   already present {skipped:>6,}")
conn.rollback(); conn.close()

THE BOEING COMPANY                       inserted      0   already present  2,914


Next: [03 · Restatements, look-ahead bias and data quality](03_restatements_and_leakage.ipynb)